In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = load_iris(as_frame=True)

X = data.data.to_numpy(dtype=float)
y = data.target.to_numpy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", data.target_names)
print("Class counts:", np.bincount(y))

Feature shape: (150, 4)
Target shape: (150,)
Classes: ['setosa' 'versicolor' 'virginica']
Class counts: [50 50 50]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (120, 4)
Testing shape: (30, 4)


In [11]:
class SoftmaxRegression:
    def __init__(self, lr=0.1, epochs=3000):
        self.lr = lr
        self.epochs = epochs
        self.loss_history = []

    def _softmax(self, logits):
        # Numerically stable softmax
        logits = logits - np.max(
            logits, axis=1, keepdims=True
        )

        exp_logits = np.exp(logits)

        return exp_logits / np.sum(
            exp_logits, axis=1, keepdims=True
        )

    def fit(self, X, y):
        n_samples, n_features = X.shape

        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # Convert class labels into one-hot vectors
        Y = np.eye(n_classes)[y]

        # Initialize parameters
        rng = np.random.default_rng(42)

        self.W = rng.normal(
            0, 0.01, size=(n_features, n_classes)
        )
        self.b = np.zeros((1, n_classes))

        for epoch in range(self.epochs):
            # 1. Forward pass
            logits = X @ self.W + self.b
            probs = self._softmax(logits)

            # 2. Cross-entropy loss
            eps = 1e-15
            loss = -np.mean(
                np.sum(Y * np.log(probs + eps), axis=1)
            )
            self.loss_history.append(loss)

            # 3. Gradients
            error = probs - Y

            dW = (X.T @ error) / n_samples
            db = np.mean(error, axis=0, keepdims=True)

            # 4. Gradient descent update
            self.W -= self.lr * dW
            self.b -= self.lr * db
  
        return self

    def predict_proba(self, X):
        logits = X @ self.W + self.b
        return self._softmax(logits)

    def predict(self, X):
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

In [12]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    log_loss
)

model = SoftmaxRegression(
    lr=0.1,
    epochs=3000
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Log loss:", log_loss(y_test, y_prob))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=data.target_names
))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9666666666666667
Log loss: 0.10372037543212091

Classification report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Confusion matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


In [13]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    solver="lbfgs",
    max_iter=3000
)

model.fit(X_train, y_train)

print("Predictions:", model.predict(X_test)[:10])
print("Probabilities:\n", model.predict_proba(X_test)[:5])
print("Accuracy:", model.score(X_test, y_test))

Predictions: [0 2 1 1 0 1 0 0 2 1]
Probabilities:
 [[9.78818005e-01 2.11816311e-02 3.63821812e-07]
 [3.79836951e-03 3.69220168e-01 6.26981463e-01]
 [1.48799040e-01 8.42474895e-01 8.72606441e-03]
 [9.54449962e-02 8.94618634e-01 9.93636953e-03]
 [9.88493051e-01 1.15067767e-02 1.72549650e-07]]
Accuracy: 0.9333333333333333
